In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


def parse_stat(text):
    if pd.isna(text): return 0.0
    if isinstance(text, (int, float)): return float(text)
    try:
        return float(re.split('±', str(text))[0])
    except:
        return 0.0


df = pd.read_csv('MIL/all_metrics_summary.csv')

target_method = 'Ensemble'
methods_list = ['AB_MIL', 'TRANS_MIL', 'CLAM_MB_MIL', 'CLAM_SB_MIL', 'WIKG_MIL',
                'MAMBA2D_MIL', 'AEM_MIL', 'MICO_MIL', 'MSM_MIL', 'TDA_MIL', 'GDF_MIL', 'Ensemble']
df = df[df['method'].isin(methods_list)].reset_index(drop=True)

perf_metrics = ['acc', 'bacc', 'macro_auc', 'macro_f1', 'macro_recall', 'quadratic_kappa']
metrics_map = {
    'acc': 'Accuracy', 'bacc': 'Bal. Acc.', 'macro_auc': 'Macro AUC',
    'macro_f1': 'Macro F1', 'macro_recall': 'Macro Recall', 'quadratic_kappa': 'Quad. Kappa'
}

colors = {
    'top1': '#D32F2F', 'top2': '#2E7D32', 'top3': '#1565C0',
    'target_out': '#5E35B1',  # 跌出前三时高亮深紫色
    'others': '#E0E4E8', 'range': '#F4F5F7'
}

fig, ax = plt.subplots(figsize=(14, 8.5), dpi=120)
sns.set_style("white")

for i, m in enumerate(perf_metrics):
    temp_df = df[['method', m]].copy()
    temp_df['val'] = temp_df[m].apply(parse_stat)

    # 严格绝对排序
    sorted_df = temp_df.sort_values('val', ascending=False).reset_index(drop=True)

    # 归一化 X 坐标映射
    v_min, v_max = sorted_df['val'].min(), sorted_df['val'].max()
    denom = (v_max - v_min) if (v_max - v_min) > 1e-9 else 1e-9
    sorted_df['plot_x'] = 0.18 + (sorted_df['val'] - v_min) / denom * 0.67

    ax.hlines(y=i, xmin=0.15, xmax=0.88, color=colors['range'], linewidth=16, zorder=1, capstyle='round')

    top3_df = sorted_df.iloc[:3]
    others_df = sorted_df.iloc[3:]

    target_in_top3 = target_method in top3_df['method'].values

    if not target_in_top3:
        target_row = others_df[others_df['method'] == target_method]
        others_df = others_df[others_df['method'] != target_method]

    ax.scatter(others_df['plot_x'], [i] * len(others_df), color=colors['others'], s=55, alpha=0.7, zorder=2)

    plot_queue = []
    for rank, (_, row) in enumerate(top3_df.iterrows()):
        plot_queue.append((f'top{rank + 1}', row))

    if not target_in_top3 and not target_row.empty:
        plot_queue.append(('target_out', target_row.iloc[0]))

    occupied_x = []

    for style_key, row in plot_queue:
        px = row['plot_x']
        val = row['val']
        name = row['method'].replace('_MIL', '')
        is_target = (row['method'] == target_method)

        ax.scatter(px, i, color=colors[style_key], s=110, edgecolors='white', linewidths=1.5, zorder=4)

        ha_style = 'center'
        text_x = px

        # 遍历检查是否与已有的文字标签 X 轴冲突
        for past_x in occupied_x:
            if abs(px - past_x) < 0.05:
                # 如果当前点在已有点的左侧，文字继续往左挪并右对齐；反之亦然
                if px < past_x:
                    text_x = px - 0.018
                    ha_style = 'right'
                else:
                    text_x = px + 0.018
                    ha_style = 'left'
                break

        occupied_x.append(text_x)

        font_w = 'bold' if is_target else 'semibold'
        ax.text(text_x, i - 0.28, f"{name}\n{val:.4f}", color=colors[style_key],
                ha=ha_style, va='top', fontsize=9, fontweight=font_w,
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.85))

ax.set_yticks(range(len(perf_metrics)))
ax.set_yticklabels([metrics_map[m] for m in perf_metrics], fontsize=12.5, fontweight='bold', color='#222222')
ax.set_xticks([])
ax.set_xticklabels([])
ax.tick_params(left=False)
sns.despine(left=True, bottom=True)

ax.set_ylim(-0.5, len(perf_metrics) - 0.2)
ax.set_xlim(0.13, 0.95)

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top1'], markersize=10, label='Top-1'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top2'], markersize=10, label='Top-2'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['top3'], markersize=10, label='Top-3'),
    # Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['target_out'], markersize=10, label=f'{target_method} (Out of Top-3)'),
    Line2D([0], [0], marker='o', color='white', markerfacecolor=colors['others'], markersize=8, label='Other Baselines')
]

leg = ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.015),
                ncol=5, frameon=True, fontsize=10.5, edgecolor='#F0F0F0', facecolor='white')
leg.get_frame().set_linewidth(1.0)
for text in leg.get_texts():
    text.set_color("#333333")
    text.set_weight("semibold")

plt.suptitle('MIL Method Comparison', fontsize=15, fontweight='bold', x=0.5, y=0.92)
plt.savefig('./result-int/mil-contrast.svg', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches


types = ['CNB', 'RP', 'TURP', 'Total']
metrics = ['acc', 'bacc', 'quadratic_kappa', 'macro_auc', 'macro_f1', 'macro_recall']
metric_labels = ['Accuracy', 'Balance\nAcc.', 'Quadratic\nKappa', 'Macro\nAUC', 'Macro\nF1', 'Macro\nRecall']


df = pd.read_csv('MIL/Ensemble/Seed/final_evaluation_summary1.csv')
data = df[['acc','bacc','quadratic_kappa','macro_auc','macro_f1','macro_recall']].values
COLOR_CNB = '#2E8B57'    # 海绿色 - 完美分类
COLOR_RP = '#4682B4'     # 钢蓝色
COLOR_TURP = '#CD853F'   # 秘鲁色 - 挑战性
COLOR_TOTAL = '#6A5ACD'  # 石板蓝

colors = [COLOR_CNB, COLOR_RP, COLOR_TURP, COLOR_TOTAL]

fig, ax = plt.subplots(figsize=(16, 9))

n_metrics = len(metrics)
n_types = len(types)
x = np.arange(n_metrics)
width = 0.18  # 柱宽

# 绘制分组柱状图
for i, (type_name, color) in enumerate(zip(types, colors)):
    offset = (i - n_types/2 + 0.5) * width
    bars = ax.bar(x + offset, data[i], width,
                  label=type_name, color=color,
                  edgecolor='white', linewidth=0.8,
                  alpha=0.9, zorder=3)

    for j, (bar, val) in enumerate(zip(bars, data[i])):
        if val == 1.0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   '1.00', ha='center', va='bottom', fontsize=8.5,
                   fontweight='bold', color=color)
        else:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   f'{val:.3f}' if val < 0.95 else f'{val:.2f}',
                   ha='center', va='bottom', fontsize=7.5, color='#333333')

for j in range(n_metrics):
    offset = (0 - n_types/2 + 0.5) * width
    x_pos = x[j] + offset
    ax.annotate('★', xy=(x_pos, 1.0), xytext=(x_pos, 1.06), fontsize=14, color=COLOR_CNB, ha='center',
                fontweight='bold')


turp_recall_idx = 5
turp_recall_val = data[2, turp_recall_idx]
offset_turp = (2 - n_types/2 + 0.5) * width
x_turp = x[turp_recall_idx] + offset_turp


ax.axhline(y=0.9, color='#CCCCCC', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.text(n_metrics - 0.5, 0.905, '0.9', fontsize=9, color='#999999', ha='right')

ax.axhline(y=0.95, color='#CCCCCC', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.text(n_metrics - 0.5, 0.955, '0.95', fontsize=9, color='#999999', ha='right')

ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Cross-Specimen Generalization: CNB, RP, and TURP', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10.5)
ax.set_ylim(0.65, 1.18)
ax.set_xlim(-0.5, n_metrics - 0.5)

# 图例
legend = ax.legend(loc='upper left', fontsize=11, frameon=True, fancybox=True, shadow=True, ncol=2)
legend.get_frame().set_facecolor('#FAFAFA')
legend.get_frame().set_edgecolor('#CCCCCC')

# 网格
ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)
ax.set_axisbelow(True)

# 背景
ax.set_facecolor('#FAFAFA')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')


plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('result-int/specimen_perf.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
